# 05 Detect service account deviations

Extra lab · about 60 minutes · after notebooks 01–04. Read [the lab guide](guides/05_Detect_service_account_deviations.md) for the brief, field contracts and practice targets. You apply the workshop method to a new detection class: map AWS CloudTrail, GCP Cloud Audit Logs and Azure Activity Log records to one contract, learn what each service account normally does, and flag what it has never done before. All logs are synthetic and local. Edit `labs/service_account_baseline/detection.sql`.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'labs/service_account_baseline/detection.sql').exists()), None)
if root is None:
    raise FileNotFoundError('Open this notebook inside the extracted workshop repository.')
sys.path.insert(0, str(root))
from workshop.service_account_baseline import Lab, rows

# A fresh temporary database per attempt. The raw logs are loaded once.
attempt = tempfile.TemporaryDirectory(prefix='service_account_baseline-notebook-')
lab = Lab(Path(attempt.name) / 'state.duckdb')
detection = root / 'labs/service_account_baseline/detection.sql'
lab.window()

## 1. Read the raw records

Each cloud names the same ideas differently. Find the caller, the account, the source address, the client and the outcome in each record. AWS is mapped for you in `detection.sql` as a worked example. Note what is missing: Azure Activity Log has no user agent.

In [ ]:
for provider in ('AWS', 'GCP', 'Azure'):
    print(f'--- {provider}')
    print(json.dumps(lab.sample(provider, 'backup-agent', '192.0.2.10'), indent=2))

## 2. Map GCP and Azure to the shared contract

Complete the `TODO` expressions in `sa.gcp_events` and `sa.azure_events`, then rerun this cell. Every provider should have zero missing principals and accounts, and service-account rows should dominate. Azure `Start` events are neither success nor failure.

In [ ]:
lab.run(detection)
lab.quality()

## 3. Baseline and deviations

The starter rule is the simple version from the article: compare exact values against a baseline keyed by principal. Inspect what it reports, then fix `sa.observations`, `sa.baseline` and `sa.deviations` following step 3 of the guide.

In [ ]:
rows(lab.db, "SELECT * FROM sa.deviations ORDER BY cloud_provider, principal, dimension, observed_value")

Accounts without enough history cannot be baselined. They belong in a separate review list, not in the deviation alerts and not silently dropped.

In [ ]:
rows(lab.db, "SELECT * FROM sa.insufficient_baseline ORDER BY cloud_provider, principal")

## 4. Triage

For each principal with deviations, decide: credential misuse, legitimate change, or unknown. Use the evidence query below. Then answer: which attack in this data set does the rule not see, and what other evidence would reveal it?

In [ ]:
rows(lab.db, """SELECT e.cloud_provider, e.principal, e.time, e.cloud_api_action, e.action_type,
       e.source_ip, e.user_agent, e.status_id
FROM sa.events e JOIN sa.deviations d USING (cloud_provider, account_uid, principal)
WHERE e.time >= (SELECT detection_start FROM lab.detection_window)
GROUP BY ALL ORDER BY e.cloud_provider, e.principal, e.time""")

## Submit

Save `detection.sql` and a short note: your identity and network choices, one false positive you removed, one alert you consider benign and why, and the blind spot. Practice targets are in the guide. The organizer grades with a different data set, so check that your SQL follows the contract rather than the counts.

When finished, remove only this attempt’s temporary database.

In [ ]:
attempt.cleanup()